# 11 · Generate synchronized XDF

Combine raw XDFs in `public_raw_data_root` (from `Anonymization/10.5`) with synchronized CSVs from step 10. ET also needs manually staged `preprocessed_data_root/ET` TSVs and mapping 01.

BEFORE RUN THIS the Anonymization/10.5 NEED TO BE FINISHED.

Writes `preprocessed_data_root/XDF`.


In [1]:
import struct
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / "config.json").is_file():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError("Could not locate config.json")

from helpers.project_config import load_project_config

CONFIG = load_project_config(PROJECT_ROOT / "config.json")
RAW_ROOT = CONFIG.path("paths", "public_raw_data_root")
PROCESSED_ROOT = CONFIG.path("paths", "preprocessed_data_root")
OUTPUT_ROOT = PROCESSED_ROOT / "XDF"
ET_MAPPING_CSV = CONFIG.path("files", "et_lsl_alignment_csv")
CHUNK_ROWS = 1000
OVERWRITE = True


In [2]:
def varlen(value):
    width = 1 if value < 256 else 4 if value < 2**32 else 8
    return bytes([width]) + int(value).to_bytes(width, "little")


def write_chunk(handle, tag, payload):
    handle.write(varlen(len(payload) + 2) + struct.pack("<H", tag) + payload)


def xdf_chunks(path, metadata_only=False):
    with path.open("rb") as handle:
        size = path.stat().st_size
        if handle.read(4) != b"XDF:":
            raise ValueError(f"Not an XDF file: {path}")
        while width := handle.read(1):
            if width[0] not in (1, 4, 8):
                raise ValueError(f"Invalid XDF chunk length: {path}")
            length_bytes = handle.read(width[0])
            if len(length_bytes) != width[0]:
                raise ValueError(f"Truncated XDF length: {path}")
            length = int.from_bytes(length_bytes, "little")
            if length < 2 or handle.tell() + length > size:
                raise ValueError(f"Truncated XDF chunk: {path}")
            tag_bytes = handle.read(2)
            tag = int.from_bytes(tag_bytes, "little")
            if metadata_only and tag not in (1, 2, 6):
                handle.seek(length - 2, 1)
                continue
            payload = handle.read(length - 2)
            yield tag, payload, width + length_bytes + tag_bytes + payload


def stream_header(part, key, source_name):
    info = ET.Element("info")
    fields = {
        "name": part["name"], "type": part["type"],
        "channel_count": len(part["columns"]), "nominal_srate": 0,
        "channel_format": part["format"],
        "source_id": f"{key}:{part['name']}",
    }
    for name, value in fields.items():
        ET.SubElement(info, name).text = str(value)
    desc = ET.SubElement(info, "desc")
    ET.SubElement(desc, "source_file").text = source_name
    ET.SubElement(desc, "timestamp_clock").text = "LSL recorder clock, seconds"
    ET.SubElement(desc, "missing_values").text = (
        "NaN for numeric channels; empty string for text channels"
    )
    channels = ET.SubElement(desc, "channels")
    for column in part["columns"]:
        ET.SubElement(ET.SubElement(channels, "channel"), "label").text = column
    return struct.pack("<I", part["id"]) + ET.tostring(info, encoding="utf-8")


def write_samples(handle, part, table, timestamps):
    payload = bytearray(struct.pack("<I", part["id"]) + varlen(len(timestamps)))
    values = table[part["columns"]]
    if part["format"] == "double64":
        samples = np.empty(len(timestamps), dtype=[
            ("width", "u1"), ("timestamp", "<f8"),
            ("values", "<f8", (len(part["columns"]),)),
        ])
        samples["width"] = 8
        samples["timestamp"] = timestamps
        samples["values"] = values.replace("", "nan").to_numpy(dtype="<f8")
        payload.extend(samples.tobytes())
    else:
        for timestamp, row in zip(timestamps, values.itertuples(index=False, name=None)):
            payload.extend(struct.pack("<Bd", 8, timestamp))
            for value in row:
                encoded = value.encode("utf-8")
                payload.extend(varlen(len(encoded)))
                payload.extend(encoded)
    write_chunk(handle, 3, payload)


def stream_footer(part, count, first, last):
    info = ET.Element("info")
    ET.SubElement(info, "sample_count").text = str(count)
    if count:
        ET.SubElement(info, "first_timestamp").text = repr(first)
        ET.SubElement(info, "last_timestamp").text = repr(last)
    return struct.pack("<I", part["id"]) + ET.tostring(info, encoding="utf-8")


In [3]:
TEXT_COLUMNS = {
    "IDUN": set(),
    "DriverVideo": set(),
    "Eyeball": {"Left_Confidence", "Right_Confidence"},
    "ET": {
        "Sensor", "Project name", "Participant name",
        "Recording Fixation filter name", "Event", "Event value",
        "Validity left", "Validity right", "Recording media name",
        "Eye movement type", "Ungrouped",
    },
}


def processed_tables(key, et_mappings):
    pid, visit = key.split("_", 1)
    tables = []
    sources = [
        ("IDUN", "EEG", f"IDUN/{key}_idun_synchronized.csv"),
        ("DriverVideo", "Landmarks", f"DriverVideo/{key}_driver_video_synchronized.csv"),
        ("Eyeball", "Pupil", f"Eyeball/{key}_eyeball_synchronized.csv"),
        ("ET", "EyeTracking", f"ET/{key}.tsv"),
    ]
    for name, stream_type, relative in sources:
        path = PROCESSED_ROOT / relative
        if not path.is_file():
            print(f"{key}: {name} skipped (processed file absent)")
            continue
        mapping = None
        if name == "ET":
            rows = et_mappings[
                (et_mappings["pid"] == pid) & (et_mappings["visit"] == visit)
            ]
            if len(rows) > 1:
                raise ValueError(f"{key}: ambiguous ET/LSL mapping")
            if rows.empty or str(rows.iloc[0]["usable"]).lower() != "true":
                print(f"{key}: ET skipped (no usable Step 01 mapping)")
                continue
            a, b = rows.iloc[0][["intercept_s", "scale"]].astype(float)
            if not np.isfinite([a, b]).all() or b <= 0:
                raise ValueError(f"{key}: invalid ET/LSL coefficients")
            mapping = (a, b)
        sep = "\t" if name == "ET" else ","
        columns = pd.read_csv(path, sep=sep, nrows=0).columns.tolist()
        timestamp_column = "Recording timestamp" if name == "ET" else "lsl_timestamp"
        if timestamp_column not in columns:
            raise ValueError(f"{path}: missing {timestamp_column}")
        parts = []
        for is_text in (False, True):
            selected = [c for c in columns if (c in TEXT_COLUMNS[name]) == is_text]
            if selected:
                parts.append({
                    "name": name + ("_labels" if is_text else ""),
                    "type": "Labels" if is_text else stream_type,
                    "format": "string" if is_text else "double64",
                    "columns": selected,
                })
        tables.append({
            "name": name, "path": path, "sep": sep,
            "mapping": mapping, "parts": parts,
        })
    return tables


def append_table(handle, spec):
    count, total = 0, 0
    first = last = None
    reader = pd.read_csv(
        spec["path"], sep=spec["sep"], dtype=str, keep_default_na=False,
        chunksize=CHUNK_ROWS,
    )
    with reader:
        for table in reader:
            total += len(table)
            if spec["mapping"] is None:
                timestamps = table["lsl_timestamp"].replace("", "nan").to_numpy(float)
                keep = np.isfinite(timestamps)
            else:
                a, b = spec["mapping"]
                et_seconds = table["Recording timestamp"].replace("", "nan").to_numpy(float) / 1e6
                timestamps = a + b * et_seconds
                keep = np.isfinite(et_seconds) & np.isfinite(timestamps)
            table, timestamps = table.loc[keep], timestamps[keep]
            if not len(timestamps):
                continue
            if np.any(np.diff(timestamps) < 0) or (last is not None and timestamps[0] < last):
                raise ValueError(f"Decreasing LSL timestamps: {spec['path']}")
            if first is None:
                first = float(timestamps[0])
                for part in spec["parts"]:
                    write_chunk(handle, 4, struct.pack("<Idd", part["id"], first, 0.0))
            for part in spec["parts"]:
                write_samples(handle, part, table, timestamps)
            last = float(timestamps[-1])
            count += len(timestamps)
    print(f"  {spec['name']}: {count:,} rows written; {total - count:,} with missing/invalid LSL time")
    return [stream_footer(part, count, first, last) for part in spec["parts"]]


In [4]:
def generate_xdf(source, target, key, tables):
    metadata = list(xdf_chunks(source, metadata_only=True))
    headers = [(payload, raw) for tag, payload, raw in metadata if tag == 2]
    file_headers = [raw for tag, payload, raw in metadata if tag == 1]
    if len(file_headers) != 1 or not headers:
        raise ValueError(f"Missing/duplicate XDF file header or no stream headers: {source}")
    ids, removed = set(), set()
    for payload, raw in headers:
        stream_id = struct.unpack_from("<I", payload)[0]
        if stream_id in ids:
            raise ValueError(f"Duplicate XDF stream ID: {source}")
        ids.add(stream_id)
        if ET.fromstring(payload[4:]).findtext("name", "").strip().casefold() == "idun":
            removed.add(stream_id)
    next_id = max(ids) + 1
    for spec in tables:
        for part in spec["parts"]:
            part["id"] = next_id
            next_id += 1
    original_footers = [
        raw for tag, payload, raw in metadata
        if tag == 6 and struct.unpack_from("<I", payload)[0] not in removed
    ]
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(".xdf.part")
    handle = temporary.open("xb")
    try:
        with handle:
            handle.write(b"XDF:")
            handle.write(file_headers[0])
            for payload, raw in headers:
                if struct.unpack_from("<I", payload)[0] not in removed:
                    handle.write(raw)
            for spec in tables:
                for part in spec["parts"]:
                    write_chunk(handle, 2, stream_header(part, key, spec["path"].name))
            for tag, payload, raw in xdf_chunks(source):
                if tag in (1, 2, 6):
                    continue
                if tag in (3, 4) and struct.unpack_from("<I", payload)[0] in removed:
                    continue
                handle.write(raw)
            added_footers = []
            for spec in tables:
                added_footers.extend(append_table(handle, spec))
            for raw in original_footers:
                handle.write(raw)
            for payload in added_footers:
                write_chunk(handle, 6, payload)
        if OVERWRITE:
            temporary.replace(target)
        else:
            temporary.rename(target)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise
    print(f"Saved {target.name}; removed {len(removed)} original IDUN stream(s)")


et_mappings = (
    pd.read_csv(ET_MAPPING_CSV, dtype=str, keep_default_na=False)
    if ET_MAPPING_CSV.is_file()
    else pd.DataFrame(columns=["pid", "visit", "usable"])
)
sources = sorted(RAW_ROOT.glob("*/LSL/*.xdf"))
if not sources:
    raise FileNotFoundError(f"No source XDF files under {RAW_ROOT}")
jobs = {}
for source in sources:
    pid, visit, _ = source.stem.split("_", 2)
    key = f"{pid}_{visit}"
    if pid != source.parent.parent.name or key in jobs:
        raise ValueError(f"Ambiguous recording identity: {source}")
    jobs[key] = source

for key, source in jobs.items():
    target = OUTPUT_ROOT / f"{key}_synchronized.xdf"
    if target.exists() and not OVERWRITE:
        print(f"{key}: existing output skipped")
        continue
    tables = processed_tables(key, et_mappings)
    print(f"{key}: generating XDF")
    generate_xdf(source, target, key, tables)


A7G5SM_VX: IDUN skipped (processed file absent)
A7G5SM_VX: generating XDF


  DriverVideo: 13,653 rows written; 0 with missing/invalid LSL time


  Eyeball: 110,750 rows written; 0 with missing/invalid LSL time


  ET: 783,694 rows written; 0 with missing/invalid LSL time
Saved A7G5SM_VX_synchronized.xdf; removed 0 original IDUN stream(s)
ABVLGQ_VX: generating XDF


  IDUN: 739,720 rows written; 0 with missing/invalid LSL time


...

  ET: 826,689 rows written; 0 with missing/invalid LSL time
Saved Z8HDMS_VX_synchronized.xdf; removed 1 original IDUN stream(s)
ZQTXLD_VX: generating XDF


  IDUN: 689,360 rows written; 0 with missing/invalid LSL time


  DriverVideo: 14,568 rows written; 0 with missing/invalid LSL time


  Eyeball: 143,100 rows written; 0 with missing/invalid LSL time


  ET: 853,799 rows written; 0 with missing/invalid LSL time
Saved ZQTXLD_VX_synchronized.xdf; removed 1 original IDUN stream(s)
